# 01. 첫 실행: `run` → `diff` → `apply`

이 노트북은 syncmill을 처음 쓰는 분이 **위에서 아래로 셀을 하나씩 실행하면서** 기본 흐름을
몸으로 익히는 실습입니다. 명령마다 "무엇을 하는지"뿐 아니라 **왜 그렇게 하는지, 실제
프로젝트에서는 어떻게 쓰는지**를 함께 설명합니다.

**이런 분께 맞습니다**

- Claude Code, Codex CLI, Kimi Code 같은 AI 코딩 CLI를 하나쯤 써 봤거나 써 보려는 분
- 여러 AI 에이전트를 함께 쓰고 싶지만, 내 코드가 엉망이 될까 걱정되는 분
- 문서를 읽기 전에 일단 직접 돌려 보며 감을 잡고 싶은 분

Python이나 Jupyter를 잘 몰라도 됩니다. 셀을 선택하고 `Shift + Enter`만 누르면 됩니다.

## syncmill은 어떤 문제를 푸나요?

AI 코딩 에이전트를 하나만 쓸 때는 보통 이렇게 일합니다. 터미널에서 에이전트를 열고, 요청을
적고, 에이전트가 **내 작업 폴더의 파일을 직접** 고치는 것을 지켜봅니다.

에이전트를 여러 개 쓰거나 일을 더 많이 맡기기 시작하면 불편한 점이 생깁니다.

| 겪는 문제 | syncmill의 해결 방법 |
|---|---|
| 에이전트가 내 작업 폴더를 바로 고쳐서, 마음에 안 들면 되돌리기 번거롭다 | 에이전트는 **별도의 작업용 복사본**에서 실행되고, 거기서 생긴 변경은 **패치로 보관**됩니다. syncmill은 이 패치를 내 폴더에 자동으로 적용하지 않고, 내가 `apply`할 때만 반영합니다 |
| 한 에이전트가 막히거나 사용량 한도에 걸리면 다른 CLI를 열고 요청을 다시 적어야 한다 | 실패하면 **우선순위에 따라 다음 에이전트로 자동으로 넘어갑니다**(이 노트북 9장에서 직접 봅니다) |
| 같은 일을 여러 에이전트에게 시켜 비교하고 싶지만 터미널과 폴더 관리가 번거롭다 | 명령 한 줄로 여러 에이전트를 **동시에**, 각자의 폴더에서 돌리고 하나를 고릅니다(`compete` 등 전략) |
| 어제 한 에이전트가 알아낸 내용을 오늘 다른 에이전트는 모른다 | 성공한 결과를 **공유 메모리**에 남기고, 다음 작업의 프롬프트에 관련 기록을 붙여 줍니다(8장) |

**언제는 굳이 필요 없을까요?** 에디터 안에서 한두 줄을 고치거나, 에이전트와 대화하며 탐색하는
정도라면 에이전트 CLI를 그냥 쓰는 편이 간단합니다. syncmill은 "작업을 맡기고, 결과를
검토해서, 받아들일지 정하는" 흐름이 필요할 때 빛을 봅니다.

## 이 노트북에서 해 보는 것

| 장 | 하는 일 | 끝나면 알게 되는 것 |
|---|---|---|
| 1~2 | 준비 확인, 연습용 샌드박스 만들기 | 안전하게 실험하는 방법 |
| 3 | 에이전트를 직접 실행해 보기 | syncmill이 에이전트를 어떻게 부르는지 |
| 4~5 | `init`, `doctor`, `run --dry-run` | 실행 전에 비용 없이 점검하는 습관 |
| 6~7 | `run` → `diff` → `apply`, 실행 기록 보기 | 맡기고, 검토하고, 받아들이는 기본 흐름과 출력 읽는 법 |
| 8 | 두 번째 에이전트 실행 | 공유 메모리가 에이전트끼리 결과를 이어 주는 방식 |
| 9 | 실패시키기, 적용 거부, 결과 버리기 | 실패·대체 실행, `apply`가 거부하는 경우, `discard` |
| 10 | 실무 응용 ①: 품질 게이트 + 대체 실행 | 테스트를 통과한 결과만 받는 방법 |
| 11 | 실무 응용 ②: `continue` | 결과를 이어서 다듬고 누적 diff를 적용하는 방법 |
| 12 | (선택) 진짜 에이전트 한 번 | 실제 실행에서 달라지는 점 |
| 13 | 내 프로젝트에 적용하기 | 실전 체크리스트와 자주 막히는 곳 |

**비용과 안전**

| 구간 | 실제 AI 호출 | 비용 |
|---|---|---|
| 1~11장, 13장 | 없음 — 가짜(fake) `codex`/`kimi` 프로그램 사용 | 무료, 로그인 불필요 |
| 12장 (선택) | `RUN_REAL = True`로 바꿀 때만 실제 CLI 1회 | 해당 provider 사용량 |

모든 작업은 `/tmp/syncmill-notebook.*` 아래 임시 폴더에서 일어납니다. 선택 셀인 12장을 빼면
여러분의 실제 프로젝트와 홈 디렉터리의 syncmill/memtomem 상태를 건드리지 않습니다.
소요 시간은 약 15분입니다.

## 0. 먼저 알아 둘 다섯 가지

| 용어 | 뜻 | 왜 필요한가 |
|---|---|---|
| **에이전트(agent)** | syncmill이 실행하는 AI 코딩 CLI — `claude`, `codex`, `kimi-code` | 실제 코드를 쓰는 주체. syncmill은 이들을 지휘할 뿐 직접 코드를 쓰지 않습니다 |
| **전략(strategy)** | 에이전트를 어떻게 돌릴지 — 기본값 `route`는 우선순위대로 시도해 처음 성공한 결과를 씀 | 상황마다 "빨리 하나만", "여럿 비교", "나눠서" 등 필요한 방식이 다르기 때문 |
| **작업 공간(worktree)** | 원본과 분리된 작업용 폴더(보통 에이전트마다 하나, 전략에 따라 단계끼리 이어 쓰기도 함) | 에이전트가 내 작업 폴더가 아닌 곳에서 작업을 시작하게 하려고 |
| **아티팩트(artifact)** | 실행 결과(패치 포함)를 보관한 기록. `run id`로 찾음 | 작업 공간은 실행 후 지워지므로, 결과를 나중에 검토·적용할 수 있게 남겨 둠 |
| **`apply`** | 검토한 아티팩트를 원본에 반영하는 **명시적인** 명령 | 무엇이 내 코드에 들어올지 사람이 마지막에 결정하도록 |

**왜 이렇게 나눠 놓았을까요?** syncmill이 에이전트를 부를 때는 비대화형(non-interactive)
모드로 실행합니다. 대화형으로 쓸 때처럼 파일을 고칠 때마다 사람에게 확인받지 않는다는
뜻입니다. 허용 범위는 CLI마다 다릅니다. Codex는 syncmill이 "작업 폴더 쓰기 허용, 승인 묻지 않음"
(`sandbox_mode="workspace-write"`, `approval_policy="never"`)으로 지정해 실행하고, Claude와 Kimi는
각 CLI의 설정을 따릅니다. 사람이 한 단계씩 지켜보지 않으니,
대신 **"별도 작업 폴더에서 일하게 하고 → 결과를 패치로 받아 → 사람이 검토한 뒤 반영"** 하는
구조를 씁니다. 코드 리뷰를 거쳐 PR을 머지하는 것과 같은 흐름입니다.

> **주의: 작업 폴더는 보안 샌드박스가 아닙니다.** syncmill은 에이전트를 별도 폴더에서 *시작*시키고
> 그 폴더의 변경만 패치로 모을 뿐, 에이전트 프로그램이 절대 경로로 다른 파일을 읽거나 쓰거나
> 네트워크에 접속하는 것까지 막지는 않습니다. 그런 제한은 각 CLI의 권한·샌드박스 설정이 맡습니다.

그래서 핵심 규칙은 하나입니다. **`syncmill run`은 원본 파일을 바꾸지 않습니다.** 원본은
여러분이 `syncmill apply <run-id>`를 실행할 때만 바뀝니다. 자세한 설명은
[사용 안내서의 "꼭 알아야 할 5가지"](../user-guide.ko.md#꼭-알아야-할-5가지)를 보세요.

## 1. 준비 확인

이 노트북은 syncmill 소스 폴더에서 다음처럼 열었다고 가정합니다.

```bash
uv sync
uv run --with jupyterlab jupyter lab docs/notebooks
```

아래 셀은 같은 폴더의 `_sandbox.py` 도우미를 불러오고 syncmill이 준비됐는지 확인합니다.
도우미는 syncmill 내부 코드를 쓰지 않습니다. 여러분이 셸에서 치는 것과 **똑같은 `syncmill`
명령**을 대신 실행해 주므로, 여기서 본 출력은 셸에서 직접 칠 때와 같습니다.

> 실제 프로젝트에서 쓸 때 필요한 것은 세 가지입니다. ① syncmill 설치, ② Git 저장소,
> ③ 에이전트 CLI 하나 이상 설치·로그인. 이 노트북에서는 ③을 가짜 CLI로 대신합니다.

In [1]:
import subprocess
from _sandbox import Sandbox, fake_agent_source, jsonl, syncmill_executable

version = subprocess.run([syncmill_executable(), "--version"], capture_output=True, text=True)
print(version.stdout.strip())

syncmill, version 0.2.0


## 2. 일회용 샌드박스 만들기

**왜 샌드박스를 쓰나요?** 처음 배울 때는 실수해도 아무 상관 없는 곳이 좋습니다. 그래서 이
노트북은 `/tmp` 아래에 연습용 폴더를 만들고, README 하나만 커밋된 작은 Git 저장소
(`demo-repo`)를 그 안에 준비합니다. 여기서 무엇을 해도 여러분의 실제 프로젝트에는 영향이 없습니다.

> **실제 프로젝트에서는 이런 샌드박스를 따로 만들지 않습니다.** syncmill이 결과를 원본에 자동으로
> 적용하지 않으므로, 무엇을 받아들일지는 `apply` 때 여러분이 정합니다. 이 샌드박스는 "syncmill
> 설정과 상태까지 통째로 버릴 수 있게" 하려는 학습용 장치입니다. (에이전트 프로그램 자체가 무엇에
> 접근할 수 있는지는 0장의 주의 문구처럼 각 CLI 설정을 따릅니다.)

샌드박스 안의 명령은 이렇게 격리됩니다.

- `HOME`을 샌드박스의 `home/`으로 바꿉니다. syncmill과 memtomem(공유 메모리 저장소)이 `~` 아래에
  쓰는 상태가 전부 샌드박스 안에 생깁니다.
- `PATH`에는 가짜 에이전트가 든 `fakebin/`과 시스템 기본 경로만 둡니다. 컴퓨터에 진짜
  `claude`/`codex`가 설치돼 있어도 여기서는 호출되지 않으니 비용이 들 수 없습니다.
- 여러분 셸의 `SYNCMILL_*`, `MEMTOMEM_*`, `GIT_*` 설정은 넘겨주지 않습니다.

In [2]:
box = Sandbox.create()
print("샌드박스:", box.root)
box.tree()

샌드박스: /tmp/syncmill-notebook.vkjgtt0g
.syncmill-notebook
demo-repo/
  README.md
fakebin/
  codex
  git
  kimi
  python3
fakes/
  codex.py
  kimi.py
home/
scratch/
state/
tmp/


## 3. 에이전트는 "그냥 CLI"예요

syncmill은 에이전트를 특별한 방법으로 부르지 않습니다. `codex exec "<요청>" --json`처럼
**여러분이 터미널에서 칠 수 있는 명령을 실행하고, 그 출력을 읽을 뿐**입니다.

이 사실은 사용자에게 꽤 중요한 의미가 있습니다.

- **로그인과 요금은 각 CLI 계정을 따릅니다.** syncmill은 API 키를 따로 받거나 보관하지 않습니다.
  `codex`에 로그인돼 있으면 syncmill도 그 계정으로 `codex`를 실행합니다.
- **CLI가 설치돼 있지 않으면 그 에이전트는 쓸 수 없습니다.** 그래서 실행 전에 `doctor`로
  설치·로그인 상태를 확인합니다(5장).
- **같은 형식으로 출력하는 프로그램이면 무엇이든 대신할 수 있습니다.** 이 노트북이 비용 없이
  돌아가는 이유입니다.

아래는 이 노트북이 쓰는 가짜 `codex`의 Python 소스입니다(`fakes/codex.py`에 저장되고,
`fakebin/codex`라는 짧은 실행 파일이 이것을 실행합니다). 하는 일은 두 가지뿐입니다.

1. 작업 폴더에 `NOTES-codex.md` 파일을 만들고 한 줄을 씁니다.
2. 진짜 Codex CLI와 같은 모양의 JSON 이벤트를 한 줄씩 출력합니다.

In [3]:
print(fake_agent_source("codex"))

"""Fake `codex` CLI for the syncmill notebooks (agent id: codex).

It speaks just enough of the real CLI's output format for syncmill to parse it:
it writes one file into its working directory (the isolated worktree syncmill
gives it) and prints one JSON event per line on stdout.
"""
import json, os, sys, time
from pathlib import Path

AGENT_ID = 'codex'
BINARY = 'codex'
TARGET = "NOTES-" + BINARY + ".md"
args = sys.argv[1:]

if args[:2] == ["login", "status"] or args[:1] in (["--version"], ["--help"]):
    print(BINARY + " (fake) - logged in")
    sys.exit(0)

if BINARY == "codex":
    # codex exec [-m SEL] [-c K=V ...] [resume SESSION] PROMPT --json
    prompt = args[-2] if len(args) >= 2 and args[-1] == "--json" else ""
else:
    # kimi [-m SEL] -p PROMPT --output-format stream-json [--session ID]
    prompt = args[args.index("-p") + 1] if "-p" in args else ""

log_dir = os.environ.get("FAKE_PROMPT_LOG_DIR")
if log_dir:
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    name =

가짜 `codex`를 직접 실행해 봅시다. 이 실습은 Git 저장소가 아닌 `scratch/` 폴더에서 합니다.
`demo-repo`에서 실행하면 그 파일이 원본에 미리 생겨 버려서, 뒤에서 syncmill이 만든 변경과
구분할 수 없게 되기 때문입니다.

In [4]:
raw = box.run_fake("codex", "exec", "README 옆에 메모를 하나 남겨 줘", "--json")

{"type": "thread.started", "thread_id": "fake-codex-thread"}
{"type": "item.completed", "item": {"type": "agent_message", "text": "codex (fake) appended one line to NOTES-codex.md."}}
{"type": "turn.completed", "usage": {"input_tokens": 21, "output_tokens": 49}}


출력 중에서 syncmill이 **에이전트의 답변**으로 쓰는 것은 `item.completed` 이벤트 안의
`agent_message` 텍스트입니다. (`thread.started`의 id는 나중에 `syncmill continue`로 같은
대화를 이어 갈 때 쓰입니다.)

실제 CLI가 업데이트되면서 출력 형식이 바뀌면 syncmill이 답변을 읽지 못해 실패로 처리할 수
있습니다. 그런 경우가 생기면 syncmill 쪽 업데이트가 필요하다는 신호입니다.

In [5]:
for event in jsonl(raw.stdout):
    item = event.get("item", {})
    if event.get("type") == "item.completed" and item.get("type") == "agent_message":
        print("에이전트의 답변:", item["text"])

print("demo-repo에 NOTES-codex.md가 있나요?", (box.repo / "NOTES-codex.md").exists())
assert not (box.repo / "NOTES-codex.md").exists()

에이전트의 답변: codex (fake) appended one line to NOTES-codex.md.
demo-repo에 NOTES-codex.md가 있나요? False


## 4. 프로젝트 초기화: `syncmill init`

`init`은 이 저장소를 syncmill 프로젝트로 등록합니다. 저장소에 두 가지를 만듭니다.

- **`syncmill.toml`** — 프로젝트 설정(어떤 에이전트를 어떤 순서로 쓸지, 기본 전략, 검사 명령 등).
  팀과 함께 쓰는 설정이므로 보통 **커밋합니다.**
- **`.syncmill/`** — 이 컴퓨터에서만 쓰는 프로젝트 식별자(`project.json`)와 그것을 Git이 무시하게
  하는 `.gitignore`.

`init`은 아무것도 자동으로 stage하지 않습니다. 무엇을 커밋할지는 여러분이 정하라는 뜻이고,
그래서 바로 뒤의 `git status`에 새 파일로 보이는 것이 정상입니다.

`box.sm(...)`은 `syncmill ...`을 샌드박스 저장소에서 실행하는 도우미입니다. 셸이라면
`syncmill init`이라고 치는 것과 같습니다.

In [6]:
box.sm("init")
box.git("status", "--short", show=True)

Initialized syncmill project 74833be1-ad47-40c9-ab98-55249486674f
Config: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo/syncmill.toml
Project marker: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo/.syncmill/project.json
Local ignore: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo/.syncmill/.gitignore
These project files are not staged automatically.
Next: syncmill doctor
?? .syncmill/
?? syncmill.toml


<git status: exit 0>

설정 파일을 커밋해서 깨끗한 기준점을 만듭니다. `.syncmill/project.json`은 이 컴퓨터에서만
의미가 있는 값이라 `init`이 Git ignore 처리해 두었고, **커밋하지 않습니다.**

In [7]:
box.git("add", "syncmill.toml", ".syncmill/.gitignore")
box.git("commit", "-qm", "initialize syncmill")
status = box.git("status", "--short")
print("git status --short 결과:", repr(status.stdout))
assert status.stdout == ""

git status --short 결과: ''


## 5. 비용 없는 점검: `doctor`와 `run --dry-run`

실제로 일을 시키기 전에 두 가지를 확인하는 습관을 들이세요. 둘 다 에이전트를 부르지 않으므로
비용이 들지 않습니다.

**`doctor` — "지금 돌릴 수 있는 상태인가?"**
에이전트 프로그램이 있는지, 저장소를 스냅샷할 수 있는지, 결과를 저장할 수 있는지 봅니다.
로그인 상태는 Claude와 Codex만 확인하고, Kimi는 실행 파일이 있는지만 봅니다. 또 기본 `doctor`는
켜진 에이전트 중 **하나만** 준비돼도 `Ready.`로 끝나므로, 쓰려는 에이전트의 줄을 직접 확인하세요.
모든 에이전트가 준비됐는지 엄격하게 보려면 `syncmill doctor --strict`를 씁니다. 에이전트 실행 도중에 "CLI가 없음" 같은 이유로 실패하는 것보다 미리 아는 편이
훨씬 낫습니다. 이 샌드박스에서는 `claude`를 꺼 두었기 때문에 `Agent claude: missing`으로
나오고, 가짜 `codex`/`kimi`는 `fakebin/`에서 찾습니다.

In [8]:
box.sm("doctor")

Project: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo
Config version: 2
Agent claude: missing
Agent codex: /tmp/syncmill-notebook.vkjgtt0g/fakebin/codex
Agent codex authentication: ready
Agent kimi-code: /tmp/syncmill-notebook.vkjgtt0g/fakebin/kimi
Repository snapshot: ready
Working tree: clean
Run artifacts: writable (/private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs)
Project memory: writable (/tmp/syncmill-notebook.vkjgtt0g/state/memory/files)
Ready.


<syncmill doctor: exit 0>

**출력 읽는 법 — `doctor`**

| 줄 | 뜻 | 이럴 땐 |
|---|---|---|
| `Project:` | 점검한 저장소 위치 | 엉뚱한 폴더라면 `cd`가 잘못된 것 |
| `Config version:` | `syncmill.toml` 형식 버전 | 신경 쓰지 않아도 됨 |
| `Agent claude: missing` | `claude` 실행 파일을 찾지 못함 | 여기서는 일부러 뺐음. 실제로 쓸 에이전트라면 설치 필요 |
| `Agent codex: <경로>` | 찾은 실행 파일 위치 | 예상과 다른 경로라면 다른 버전이 먼저 잡힌 것 |
| `Agent codex authentication: ready` | 로그인 확인됨 (Claude·Codex만 확인) | `not authenticated` 경고면 그 CLI에서 로그인 |
| `Repository snapshot: ready` | 저장소를 스냅샷할 수 있음 | 실패하면 Git 상태(충돌·리베이스 진행 중 등)를 확인 |
| `Working tree: clean` | 커밋하지 않은 변경 없음 | `dirty (will be preserved)`면 그 변경도 스냅샷에 포함된다는 뜻 |
| `Run artifacts: writable (...)` | 실행 결과를 저장할 위치 | |
| `Project memory: writable (...)` | 공유 메모리를 저장할 위치 | |
| `Ready.` | 켜진 에이전트 중 하나 이상 준비 완료 | 쓰려는 에이전트의 줄을 따로 확인 |

**`run --dry-run` — "이대로 실행하면 무엇이 일어나나?"**
어떤 전략으로, 어떤 에이전트를, **최대 몇 번 호출할지**(`maximum_agent_calls`) 보여 주기만 하고
에이전트는 실행하지 않습니다.

이 숫자는 비용과 직결됩니다. (에이전트 호출 횟수의 상한일 뿐, 토큰이나 요금의 상한은 아닙니다.) 지금은 에이전트 하나를 지정했으니 1이지만, 여러 에이전트를
동시에 돌리거나 여러 라운드를 도는 전략은 호출 수가 몇 배로 늘어납니다. 새 전략이나 설정을
처음 쓸 때는 dry-run으로 숫자를 먼저 확인하세요.

In [9]:
TASK = "README 옆에 메모 파일을 하나 추가해 줘"
box.sm("run", "--dry-run", TASK, "--agent", "codex")

{
  "strategy": "route",
  "agents": [
    "codex"
  ],
  "phases": [
    "route"
  ],
  "maximum_agent_calls": 1,
  "selection": "n/a",
  "gate_commands": []
}
repository: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo
working_tree: clean
No agent was started and no source file was changed.


<syncmill run: exit 0>

**출력 읽는 법 — `run --dry-run`**

| 항목 | 뜻 |
|---|---|
| `strategy` | 쓸 전략. 지정하지 않으면 `syncmill.toml`의 `default_strategy`(기본 `route`) |
| `agents` | 후보 에이전트와 그 순서. `--agent`를 주면 하나만 남음 |
| `phases` | 전략이 거치는 단계. `route`는 한 단계, 여러 라운드 전략은 단계가 여러 개 |
| `maximum_agent_calls` | 에이전트 호출 횟수의 **최대값**. 실제로는 더 적을 수 있음(`route`는 성공하면 멈춤) |
| `selection` | 여러 후보 중 승자를 고르는 방식. 한 명씩 시도하는 `route`에는 해당 없음(`n/a`) |
| `gate_commands` | 결과를 검사할 명령. 빈 목록이면 검사 없이 받아들임(10장에서 설정해 봅니다) |
| `working_tree` | 원본 상태. `clean`, 또는 커밋하지 않은 변경이 있으면 `dirty (captured)`(그 변경도 스냅샷에 포함) |

마지막 줄 `No agent was started and no source file was changed.`가 "아무 에이전트도 부르지 않았다"는 확인입니다.

## 6. 작업 맡기기: `syncmill run`

이제 실제로 실행합니다. `--agent codex`는 "codex 하나만 쓰고, 실패해도 다른 에이전트로
넘어가지 말 것"이라는 뜻입니다.

실행하면 syncmill은 다음 순서로 일합니다.

1. 지금 저장소 상태를 **스냅샷**합니다. 커밋된 내용뿐 아니라 아직 커밋하지 않은 수정과 새 파일도
   포함합니다(Git이 무시하는 파일은 제외).
2. 그 스냅샷으로 에이전트 전용 작업 공간(worktree)을 만들고, 그 안에서 에이전트를 실행합니다.
3. 바뀐 내용을 패치로 만들어 **아티팩트**로 저장하고, 작업 공간은 정리합니다.

출력 맨 아래의 `[syncmill] run: ...` 줄이 이 실행의 id입니다. 이후 검토·적용은 이 id로 합니다.

In [10]:
result = box.sm("run", TASK, "--agent", "codex", "--timeout", "60")
run_id = box.run_id(result)
print("run id:", run_id)

[syncmill] Executing task 5a428c07...
[syncmill] Agent: codex (forced, no fallback)

[codex] Completed in 0.02s
[syncmill] Files changed: NOTES-codex.md

codex (fake) appended one line to NOTES-codex.md.
[syncmill] run: cf83dbf4-1767-4ec4-b991-02f2aa4b6688
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/cf83dbf4-1767-4ec4-b991-02f2aa4b6688
[syncmill] review: syncmill diff cf83dbf4-1767-4ec4-b991-02f2aa4b6688
[syncmill] apply: syncmill apply cf83dbf4-1767-4ec4-b991-02f2aa4b6688
run id: cf83dbf4-1767-4ec4-b991-02f2aa4b6688


**출력 읽는 법 — `run`**

| 줄 | 뜻 |
|---|---|
| `[syncmill] Executing task 1a2b3c4d...` | 작업(task)의 짧은 id. 아래의 run id와는 **다른 값**이니 헷갈리지 마세요 |
| `Agent: codex (forced, no fallback)` | `--agent`로 지정해 대체 실행 없이 codex만 씀. `--agent`가 없으면 `Agents: codex → kimi-code (priority fallback)`처럼 시도 순서가 나옴 |
| `[codex] Completed in 0.02s` | 결과를 낸 에이전트와 걸린 시간 |
| `Files changed: ...` | 에이전트가 **작업 공간에서** 바꾼 파일 |
| (빈 줄 다음의 문장) | 에이전트의 답변. 무엇을 했는지 스스로 설명한 내용 |
| `run: <id>` | 이 실행의 id. 이후 `diff`·`apply`·`continue`에 씀 |
| `artifact: <경로>` | 결과가 보관된 폴더(7장에서 열어 봅니다) |
| `review:` / `apply:` | 다음에 칠 명령을 그대로 알려 주는 안내 |

`Files changed: NOTES-codex.md`라고 나왔지만, 이것은 **에이전트의 작업 공간에서** 바뀐 파일
목록입니다. 원본 `demo-repo`는 아직 그대로입니다. 직접 확인해 봅시다.

In [11]:
print("원본에 NOTES-codex.md가 있나요?", (box.repo / "NOTES-codex.md").exists())
assert not (box.repo / "NOTES-codex.md").exists()

원본에 NOTES-codex.md가 있나요? False


**실제 프로젝트에서 쓸 때의 팁**

- **요청은 작고 구체적으로.** "버그 고쳐 줘"보다 "`parse_date`가 빈 문자열에서 예외를 내는 문제를
  고치고 테스트를 추가해 줘"가 검토하기도 쉽고 결과도 좋습니다.
- **처음에는 `--agent`로 하나만.** 설치·로그인·결과 형식이 모두 정상인지 한 번에 확인할 수 있고,
  기본 전략(`route`)에서는 호출도 한 번뿐입니다. (`--agent`는 쓸 에이전트를 하나로 줄일 뿐이라,
  여러 라운드를 도는 전략에서는 같은 에이전트가 여러 번 호출될 수 있습니다. dry-run으로 확인하세요.) 익숙해지면 `--agent`를 빼서 대체 실행을 쓰거나 다른 전략으로 넓히세요.
- **`--timeout`을 정해 두세요.** 에이전트가 오래 붙잡고 있는 작업을 끊는 안전장치입니다.
- **결과를 적용할 때까지는 원본 폴더를 그대로 두세요.** 에이전트는 실행 시작 시점의 스냅샷에서
  일하므로 그 사이의 내 수정과 섞이지는 않습니다. 대신 `apply`는 실행 이후 작업 트리
  (커밋, 커밋하지 않은 수정, Git이 무시하지 않는 새 파일)가 바뀌었으면 적용을 거부합니다(9장에서
  직접 봅니다). 다른 작업을 하고 싶다면 결과를 먼저
  적용하거나 버린 뒤에 하세요.

## 7. 검토하고 적용하기: `runs list` → `diff` → `apply`

이 단계가 syncmill 흐름의 핵심입니다. PR을 머지하기 전에 변경 내용을 읽는 것처럼, 에이전트가
만든 패치를 **읽고 나서** 받아들입니다.

`runs list`는 보관된 실행 목록입니다. 방금 실행은 아직 적용되지 않았으므로 `pending`
상태입니다. `diff`로 정확히 무엇이 바뀌는지 읽어 보세요.

In [12]:
box.sm("runs", "list")
box.sm("diff", run_id)

cf83dbf4-1767-4ec4-b991-02f2aa4b6688  pending      codex      2026-10-14T11:55:26.299571+00:00


diff --git a/NOTES-codex.md b/NOTES-codex.md
new file mode 100644
index 0000000..7acd1f0
--- /dev/null
+++ b/NOTES-codex.md
@@ -0,0 +1 @@
+- codex wrote this line.


<syncmill diff: exit 0>

**출력 읽는 법 — `runs list`**: 한 줄이 실행 하나이고, 열은 `run id`, 상태, 결과를 낸 에이전트, 실행 시각(UTC)입니다.

| 상태 | 뜻 | 다음 행동 |
|---|---|---|
| `pending` | 결과가 있고 아직 적용·폐기하지 않음 | `diff`로 검토 → `apply` 또는 `discard` |
| `applied` | 원본에 적용됨 | 테스트 후 커밋 |
| `failed` | 결과를 내지 못함. 이 노트북처럼 후보가 모두 실패하면 에이전트 열은 `none`(실패한 방식에 따라 에이전트 id가 나오기도 함) | 오류 메시지 확인 후 다시 실행 |
| `discarded` | 검토 후 쓰지 않기로 표시함 | 없음 |
| `superseded` | `continue`로 만든 후속 결과로 대체됨 | 후속 결과를 검토(11장) |

**출력 읽는 법 — `diff`**: Git과 같은 형식(unified diff)입니다.

- `diff --git a/<파일> b/<파일>` — 바뀐 파일 하나의 시작
- `new file mode` / `--- /dev/null` — 새로 생긴 파일(지운 파일이면 `+++ /dev/null`)
- `@@ -0,0 +1 @@` — 바뀐 위치. "원래 0줄짜리 → 새로 1줄"
- `+`로 시작하는 줄은 추가, `-`로 시작하는 줄은 삭제, 공백으로 시작하는 줄은 맥락(그대로인 줄)

내용이 괜찮으면 `apply`로 원본에 반영합니다. 적용이 끝나면 `Receipt`(영수증) 경로가 나오고,
같은 실행은 `applied` 상태가 됩니다. 영수증은 "언제 어떤 패치가 적용됐는지"의 기록입니다.

In [13]:
box.sm("apply", run_id)

applied = (box.repo / "NOTES-codex.md").read_text(encoding="utf-8")
print("적용된 파일 내용:", repr(applied))
assert applied == "- codex wrote this line.\n"

box.sm("runs", "list")
box.git("status", "--short", show=True)

Applied run cf83dbf4-1767-4ec4-b991-02f2aa4b6688
Receipt: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/cf83dbf4-1767-4ec4-b991-02f2aa4b6688/applications/20261014T115527280394Z.json
적용된 파일 내용: '- codex wrote this line.\n'


cf83dbf4-1767-4ec4-b991-02f2aa4b6688  applied      codex      2026-10-14T11:55:26.299571+00:00
?? NOTES-codex.md


<git status: exit 0>

**출력 읽는 법 — `apply`**: `Applied run <id>`는 패치가 원본에 들어갔다는 뜻이고, `Receipt:` 뒤의
경로는 적용 기록(영수증) 파일입니다. `runs list`에서 상태가 `pending` → `applied`로 바뀌었고,
`git status`의 `??`는 "Git이 아직 추적하지 않는 새 파일"이라는 뜻입니다.

이제 `NOTES-codex.md`가 원본에 생겼습니다. **syncmill은 커밋하지 않습니다.** 평소처럼 테스트를
돌려 보고, 괜찮으면 여러분이 커밋합니다. 여기서도 커밋해서 작업 트리를 깨끗하게 해 둡시다.

In [14]:
box.git("add", "NOTES-codex.md")
box.git("commit", "-qm", "add notes from codex")
assert box.git("status", "--short").stdout == ""

**diff가 마음에 들지 않았다면?**

- 적용하지 않고 버리기: `syncmill discard <run-id>` (9장에서 해 봅니다)
- 같은 에이전트 대화를 이어서 고치게 하기: `syncmill continue <run-id> "테스트도 추가해 줘"` (11장에서 해 봅니다)
- 아예 요청을 고쳐서 새로 `run`

### 실행 기록 들여다보기: `runs show`와 아티팩트 폴더

나중에 "이 변경은 어떤 요청으로, 어느 에이전트가, 언제 만들었지?"를 확인해야 할 때가 있습니다.
`syncmill runs show <run-id>`는 실행 기록(manifest)을 JSON으로 보여 줍니다. 자주 보는 항목만 골라 봅시다.

In [15]:
import json

manifest = json.loads(box.sm("runs", "show", run_id, show=False).stdout)
for key in ("state", "prompt", "strategy", "agent_id", "files_changed",
            "snapshot_commit", "source_fingerprint", "patch_sha256", "provider_session_id"):
    print(f"{key:20} {manifest[key]}")

state                applied
prompt               README 옆에 메모 파일을 하나 추가해 줘
strategy             route
agent_id             codex
files_changed        ['NOTES-codex.md']
snapshot_commit      45a4d2aa87624551f90c4b53e9d7022d3d8aa462
source_fingerprint   09ac051ba070329883888d13e35d4aa580f5aba36dc525b10d148bfd02189c53
patch_sha256         79273ca5cd9151d1343e8077fb6e62ef704dbe171c8d2745b21013a3e1eaa898
provider_session_id  fake-codex-thread


| 항목 | 뜻 | 어디에 쓰이나 |
|---|---|---|
| `state` | 지금 상태 (`runs list`의 상태와 같음) | |
| `prompt` | 여러분이 준 원래 요청 | 나중에 "무엇을 시켰는지" 확인 |
| `agent_id` | 결과를 낸 에이전트 | |
| `files_changed` | 패치가 바꾸는 파일 | |
| `snapshot_commit` | 에이전트가 일을 시작한 스냅샷 | 어느 시점 코드를 보고 만든 패치인지 |
| `source_fingerprint` | 실행 당시 원본 상태의 지문 | `apply`가 이 값으로 "그 사이 원본이 바뀌었나"를 판단(9장) |
| `patch_sha256` | 패치 내용의 지문 | 영수증과 대조해 "어떤 패치가 적용됐나" 확인 |
| `provider_session_id` | 에이전트 CLI의 대화 세션 id | `continue`가 이 대화를 이어 감(11장) |

아티팩트 폴더에는 세 파일이 있고, 적용 후에는 `applications/` 아래에 영수증이 생깁니다.

In [16]:
from pathlib import Path

artifact = Path(next(line.split("artifact: ", 1)[1] for line in result.stdout.splitlines()
                     if line.startswith("[syncmill] artifact: ")))
for path in sorted(artifact.rglob("*")):
    if path.is_file():
        print(path.relative_to(artifact))

receipt = json.loads(next(artifact.glob("applications/*.json")).read_text())
print()
print("영수증:", json.dumps(receipt, indent=2, ensure_ascii=False))
assert receipt["patch_sha256"] == manifest["patch_sha256"]

applications/20261014T115527280394Z.json
manifest.json
result.json
winner.patch

영수증: {
  "applied_at": "2026-10-14T11:55:27.280577+00:00",
  "repo_root": "/private/tmp/syncmill-notebook.vkjgtt0g/demo-repo",
  "patch_sha256": "79273ca5cd9151d1343e8077fb6e62ef704dbe171c8d2745b21013a3e1eaa898",
  "apply_id": "c8bd321c-8bd4-4654-bd2a-55c948885c46",
  "run_id": "cf83dbf4-1767-4ec4-b991-02f2aa4b6688"
}


- `manifest.json` — 위에서 본 실행 기록
- `result.json` — 에이전트 결과 요약(상태, 답변, 종료 코드, 걸린 시간)
- `winner.patch` — 실제로 적용되는 패치. `syncmill diff`가 보여 주는 내용입니다
- `applications/*.json` — 영수증. `patch_sha256`이 manifest와 같으므로 "검토한 바로 그 패치가 적용됐다"는 것을
  확인할 수 있습니다

## 8. 공유 메모리: 에이전트끼리 결과 이어 주기

AI 에이전트는 기본적으로 지난 세션을 기억하지 못합니다. 어제 codex가 알아낸 내용을 오늘
kimi-code는 모르고, 같은 설명을 매번 다시 적어야 합니다.

syncmill은 성공한 결과를 **공유 메모리**(`shared` namespace)에 기록해 두고, 다음 작업을 시작할
때 관련 있는 기록을 찾아 **프롬프트 앞에 붙여** 줍니다. 에이전트끼리 직접 대화하지 않아도,
어떤 에이전트를 쓰든 앞선 작업을 이어받을 수 있습니다.

이번에는 **다른 에이전트**(`kimi-code`)에게 작업을 시키고, 가짜 에이전트가 기록해 둔 "syncmill이
실제로 보낸 프롬프트"를 열어 봅시다.

In [17]:
second = box.sm("run", "README 옆에 메모를 하나 더 추가해 줘", "--agent", "kimi-code", "--timeout", "60")
print("--- kimi-code가 받은 프롬프트 ---")
print(box.prompts()[-1].read_text(encoding="utf-8"))

[syncmill] Executing task d3133c69...
[syncmill] Agent: kimi-code (forced, no fallback)

[kimi-code] Completed in 0.34s
[syncmill] Files changed: NOTES-kimi.md

kimi-code (fake) appended one line to NOTES-kimi.md.
[syncmill] run: 4357e49e-92c9-4c43-8187-4facfaa5efb8
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/4357e49e-92c9-4c43-8187-4facfaa5efb8
[syncmill] review: syncmill diff 4357e49e-92c9-4c43-8187-4facfaa5efb8
[syncmill] apply: syncmill apply 4357e49e-92c9-4c43-8187-4facfaa5efb8
--- kimi-code가 받은 프롬프트 ---
Context from previous work:
[/private/tmp/syncmill-notebook.vkjgtt0g/state/memory/files/2026-] **Task Result**

**Prompt:** README 옆에 메모 파일을 하나 추가해 줘

**Agent:** codex

**Output:**
codex (fake) appended one line to NOTES-codex.md.

**Files changed:** NOTES-codex.md

---

Task: README 옆에 메모를 하나 더 추가해 줘


프롬프트 앞부분의 `Context from previous work:` 아래에 **codex의 이전 결과**가 들어 있습니다.
kimi-code는 codex와 직접 이야기한 적이 없지만, syncmill이 공유 메모리에서 찾아 넘겨준 것입니다.

알아 둘 점:

- 메모리는 **도움이 되면 좋은 부가 기능**입니다. 메모리 저장소에 문제가 생겨도 syncmill은 작업을
  멈추지 않고, 노란색 안내를 보여 준 채 메모리 없이 계속 진행합니다.
- 기록은 관련도 순으로 찾아 붙입니다. 오래되거나 관련 없는 기록은 붙지 않을 수 있습니다.
- 이 샌드박스의 메모리는 임베딩 모델 없이 키워드 검색(BM25)만으로 동작합니다. 그래서 모델을
  내려받지 않습니다.
- 대화형으로 쓰는 Claude Code 세션도 memtomem MCP 도구로 `shared` 기록을 읽고 쓰며 syncmill과 협업할
  수 있습니다. 단, 두 쪽이 **같은 메모리 저장소**를 가리켜야 합니다. `init`한 프로젝트의 syncmill은
  기본적으로 저장소 안 `.syncmill/memory`에 기록하므로, 따로 설정하지 않은 memtomem MCP(기본
  `~/.memtomem`)에서는 이 기록이 보이지 않습니다. 협업 방법은 [사용 안내서](../user-guide.ko.md)의
  대화형 협업 부분을 보세요.

## 9. 실패하면 어떻게 되나요?

실제로 쓰다 보면 에이전트가 실패할 때가 있습니다. 로그인이 풀렸거나, 사용량 한도에 걸렸거나,
시간 안에 끝내지 못한 경우입니다. 이때 syncmill이 어떻게 동작하는지 미리 보아 두면 당황하지
않습니다.

가짜 에이전트는 환경 변수 `FAKE_CODEX_FAIL=1`이 있으면 일부러 실패합니다. 먼저 `--agent codex`로
codex 하나만 지정한 상태에서 실패시켜 봅시다. 실패를 보는 것이 목적이므로 `check=False`로
"실패해도 멈추지 말 것"을 지정합니다.

In [18]:
failed = box.sm("run", "README에 한 줄 추가해 줘", "--agent", "codex",
                check=False, extra_env={"FAKE_CODEX_FAIL": "1"})
print("종료 코드:", failed.returncode)
assert failed.returncode == 1
box.sm("runs", "list")

[syncmill] Executing task d1a8b527...
[syncmill] Agent: codex (forced, no fallback)

[syncmill] Failed
All agents failed:
  [codex] codex (fake): simulated failure
[syncmill] run: df61caa9-5324-43b1-87f1-ae8b668d0137
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/df61caa9-5324-43b1-87f1-ae8b668d0137
(exit code 1)
종료 코드: 1


df61caa9-5324-43b1-87f1-ae8b668d0137  failed       none       2026-10-14T11:55:30.165600+00:00
4357e49e-92c9-4c43-8187-4facfaa5efb8  pending      kimi-code  2026-10-14T11:55:28.742426+00:00
cf83dbf4-1767-4ec4-b991-02f2aa4b6688  applied      codex      2026-10-14T11:55:26.299571+00:00


<syncmill runs: exit 0>

**출력 읽는 법 — 실패**

- `[syncmill] Failed` / `All agents failed:` — 후보 에이전트가 모두 실패했다는 뜻입니다.
- `[codex] codex (fake): simulated failure` — `[에이전트] 실패 이유` 형식입니다. 진짜 CLI라면 여기에 로그인
  오류, 사용량 초과, `Timed out after ...` 같은 메시지가 나옵니다. **해결의 실마리는 대부분 이 줄에 있습니다.**
- 실패해도 `run:` id는 발급되지만 `review:`·`apply:` 안내는 없습니다. 적용할 패치가 없기 때문입니다.

정리하면:

- 종료 코드가 1이고 `All agents failed`와 함께 **실패 이유**(에이전트가 남긴 오류 메시지)가 나옵니다.
- 실행 기록은 `failed`로 남고, 적용할 패치가 없으므로 원본은 그대로입니다.
- `--agent`로 하나만 지정했기 때문에 다른 에이전트로 넘어가지 않았습니다.

이번에는 `--agent` **없이** 같은 상황을 만들어 봅시다. 기본 전략 `route`는 우선순위 목록의
에이전트를 차례로 시도합니다. 이 샌드박스의 순서는 `codex → kimi-code`입니다(`claude`는 꺼 둠).

In [19]:
fallback = box.sm("run", "README에 한 줄 추가해 줘", extra_env={"FAKE_CODEX_FAIL": "1"})
fallback_id = box.run_id(fallback)

[syncmill] Executing task 2e5cd6e7...
[syncmill] Agents: codex → kimi-code (priority fallback)

[kimi-code] Completed in 0.03s
[syncmill] Files changed: NOTES-kimi.md

kimi-code (fake) appended one line to NOTES-kimi.md.
[syncmill] run: b6b1fce4-2049-4fc6-a609-33c534721d80
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/b6b1fce4-2049-4fc6-a609-33c534721d80
[syncmill] review: syncmill diff b6b1fce4-2049-4fc6-a609-33c534721d80
[syncmill] apply: syncmill apply b6b1fce4-2049-4fc6-a609-33c534721d80


`Agents: codex → kimi-code (priority fallback)` 줄이 시도 순서를 보여 주고, codex가 실패하자
**kimi-code가 이어서 성공**했습니다. 한 CLI가 한도에 걸려도 작업이 멈추지 않게 하는 것이 대체
실행(fallback)의 쓰임새입니다. 성공하면 바로 멈추므로 여러 에이전트를 한꺼번에 돌리는 것보다
사용량도 아낄 수 있습니다.

### 적용이 거부되는 경우

실행이 끝난 뒤 원본 폴더를 고치면 어떻게 될까요? README를 살짝 고친 다음(커밋하지 않음) 방금 결과를
적용해 봅시다.

In [20]:
(box.repo / "README.md").write_text("# Notebook demo\n내가 직접 고친 줄\n", encoding="utf-8")
refused = box.sm("apply", fallback_id, check=False)
print("종료 코드:", refused.returncode)
assert refused.returncode == 1
print("원본에 NOTES-kimi.md가 생겼나요?", (box.repo / "NOTES-kimi.md").exists())
assert not (box.repo / "NOTES-kimi.md").exists()

Error: working tree changed since the run; review the diff and rerun
(exit code 1)
종료 코드: 1
원본에 NOTES-kimi.md가 생겼나요? False


`working tree changed since the run`이라며 **적용을 거부**했고 원본에는 아무것도 들어오지 않았습니다.
에이전트는 실행 시점의 코드를 보고 패치를 만들었으므로, 그 사이 바뀐 코드에 그대로 덮어쓰면
여러분의 수정을 망가뜨릴 수 있기 때문입니다. 바뀐 파일이 패치와 상관없는 README여도 거부합니다. (Git이 무시하는 파일의 변경은 비교에 들어가지
않습니다.)
이럴 때는 `diff`를 다시 보고, 필요하면 지금 상태에서 `run`을 다시 실행하세요.

README를 원래대로 되돌려 둡니다.

In [21]:
box.git("checkout", "--", "README.md")
print("README 되돌림:", repr((box.repo / "README.md").read_text(encoding="utf-8")))

README 되돌림: '# Notebook demo\n'


### 쓰지 않을 결과 버리기: `discard`

이 결과는 앞에서 이미 비슷한 메모를 적용했으니 필요 없다고 해 봅시다. 쓰지 않을 결과는
`discard`로 **"검토했고 적용하지 않음"** 이라고 표시해 둡니다. 나중에 목록을 볼 때 아직 검토하지
않은 `pending`과 구분할 수 있습니다.

In [22]:
box.sm("discard", fallback_id)
listing = box.sm("runs", "list").stdout
status_of = {line.split()[0]: line.split()[1] for line in listing.splitlines() if line.strip()}
print("방금 실행의 상태:", status_of[fallback_id])
assert status_of[fallback_id] == "discarded"

Discarded run b6b1fce4-2049-4fc6-a609-33c534721d80


b6b1fce4-2049-4fc6-a609-33c534721d80  discarded    kimi-code  2026-10-14T11:55:31.444950+00:00
df61caa9-5324-43b1-87f1-ae8b668d0137  failed       none       2026-10-14T11:55:30.165600+00:00
4357e49e-92c9-4c43-8187-4facfaa5efb8  pending      kimi-code  2026-10-14T11:55:28.742426+00:00
cf83dbf4-1767-4ec4-b991-02f2aa4b6688  applied      codex      2026-10-14T11:55:26.299571+00:00
방금 실행의 상태: discarded


## 10. 실무 응용 ①: 테스트를 통과한 결과만 받기

지금까지는 메모 파일을 만드는 연습이었습니다. 이번에는 실제 프로젝트와 비슷한 상황을 만들어
봅시다.

> **상황**: 팀 저장소의 `calc.py`에 있는 `average()`가 빈 리스트를 받으면 `ZeroDivisionError`로
> 죽는 버그가 있습니다. 팀에는 이미 동작을 확인하는 검사 스크립트 `check_calc.py`가 있습니다.
> 에이전트에게 버그를 고치게 하되, **검사를 통과한 결과만** 받고 싶습니다.

이럴 때 쓰는 것이 **품질 게이트(quality gate)**입니다. `syncmill.toml`의 `[gate]`에 검사 명령을 적어
두면, syncmill은 에이전트가 끝난 뒤 **그 에이전트의 작업 공간에서** 명령을 실행하고, 모두 통과(종료
코드 0)해야 성공으로 칩니다. 통과하지 못한 결과는 실패로 처리되므로, `route`라면 다음 에이전트로
넘어갑니다.

먼저 저장소에 버그가 있는 코드와 검사 스크립트를 넣고, 검사가 실제로 실패하는지 봅시다.

In [23]:
(box.repo / "calc.py").write_text(
    "def average(values):\n"
    "    return sum(values) / len(values)\n",
    encoding="utf-8",
)
(box.repo / "check_calc.py").write_text(
    "from calc import average\n\n"
    "assert average([2, 4]) == 3\n"
    "assert average([]) == 0, 'empty list must average to 0'\n"
    "print('check_calc: all good')\n",
    encoding="utf-8",
)
(box.repo / ".gitignore").write_text("__pycache__/\n", encoding="utf-8")

before_fix = box.cmd("python3", "check_calc.py")
assert before_fix.returncode != 0

Traceback (most recent call last):
  File "/private/tmp/syncmill-notebook.vkjgtt0g/demo-repo/check_calc.py", line 4, in <module>
    assert average([]) == 0, 'empty list must average to 0'
           ^^^^^^^^^^^
  File "/private/tmp/syncmill-notebook.vkjgtt0g/demo-repo/calc.py", line 2, in average
    return sum(values) / len(values)
           ~~~~~~~~~~~~^~~~~~~~~~~~~
ZeroDivisionError: division by zero
(exit code 1)


`ZeroDivisionError` — 버그가 재현됩니다.

`.gitignore`에 `__pycache__/`를 넣은 이유도 실무에서 중요합니다. 게이트가 Python을 실행하면 작업 공간에
`__pycache__/*.pyc` 같은 부산물이 생기는데, Git이 무시하지 않는 파일은 **에이전트의 변경으로 함께 패치에
담깁니다**. 결과의 `Files changed`에 예상하지 못한 파일이 보이면 `.gitignore`부터 확인하세요.

이제 `syncmill.toml`에 게이트를 설정하고, 팀원과 같은 설정을 쓰도록 커밋합니다.

In [24]:
toml_path = box.repo / "syncmill.toml"
toml_path.write_text(
    toml_path.read_text(encoding="utf-8").replace(
        "commands = []", 'commands = ["python3 check_calc.py"]'
    ),
    encoding="utf-8",
)
print(toml_path.read_text(encoding="utf-8"))

box.git("add", "calc.py", "check_calc.py", ".gitignore", "syncmill.toml")
box.git("commit", "-qm", "add calc with a check and a syncmill gate")

config_version = 2
repo_root = "."
default_strategy = "route"

[project]
id = "74833be1-ad47-40c9-ab98-55249486674f"
memory_dir = ".syncmill/memory"

[gate]
commands = ["python3 check_calc.py"]



<git commit: exit 0>

이번 실습에서는 결과를 비교하기 위해 가짜 에이전트의 행동을 **미리 정해 둡니다**. 진짜 에이전트라면 각자
알아서 코드를 고치겠지만, 여기서는 이렇게 연출합니다.

- `codex`: 빈 리스트일 때 `None`을 돌려주도록 고침 → 검사(`== 0`)를 **통과하지 못함**
- `kimi-code`: 빈 리스트일 때 `0`을 돌려주도록 고침 → 검사를 **통과함**

In [25]:
WRONG_FIX = "def average(values):\n    if not values:\n        return None\n    return sum(values) / len(values)\n"
RIGHT_FIX = "def average(values):\n    if not values:\n        return 0\n    return sum(values) / len(values)\n"

box.set_scenario({
    "codex": {"files": {"calc.py": WRONG_FIX}, "message": "Return None for an empty list."},
    "kimi-code": {"files": {"calc.py": RIGHT_FIX}, "message": "Return 0 for an empty list."},
})

FIX_TASK = "average([])가 ZeroDivisionError를 내지 않고 0을 반환하도록 calc.py를 고쳐 줘"
box.sm("run", "--dry-run", FIX_TASK)

{
  "strategy": "route",
  "agents": [
    "codex",
    "kimi-code"
  ],
  "phases": [
    "route"
  ],
  "maximum_agent_calls": 2,
  "selection": "n/a",
  "gate_commands": [
    "python3 check_calc.py"
  ]
}
repository: /private/tmp/syncmill-notebook.vkjgtt0g/demo-repo
working_tree: clean
No agent was started and no source file was changed.


<syncmill run: exit 0>

dry-run 결과를 읽어 보면 앞에서와 두 군데가 다릅니다.

- `agents`가 `codex`, `kimi-code` 두 개이고 `maximum_agent_calls`가 **2**입니다. `--agent`를 주지 않았으니
  codex가 실패하면 kimi-code까지 시도할 수 있다는 뜻입니다.
- `gate_commands`에 방금 설정한 `python3 check_calc.py`가 들어 있습니다.

먼저 게이트에 걸리면 어떤 출력이 나오는지 보기 위해, codex 하나만 지정해 실행합니다.

In [26]:
gated = box.sm("run", FIX_TASK, "--agent", "codex", check=False)
assert gated.returncode == 1

[syncmill] Executing task ec0e452b...
[syncmill] Agent: codex (forced, no fallback)

[syncmill] Failed
All agents failed:
  [codex] quality gate failed: 0/1 checks passed; failed: python3 check_calc.py
[syncmill] run: 59d84f72-32fd-4b21-8ae2-1b0286fa46b4
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/59d84f72-32fd-4b21-8ae2-1b0286fa46b4
(exit code 1)


**출력 읽는 법 — 게이트 실패**: `[codex] quality gate failed: 0/1 checks passed; failed: python3 check_calc.py`

- codex는 정상적으로 실행을 마쳤지만, 결과가 검사 1개 중 0개를 통과해서 **실패로 처리**됐습니다.
- `failed: ...` 뒤에 어떤 검사 명령이 실패했는지 나옵니다.
- 게이트 실패는 "에이전트가 고장 났다"가 아니라 "결과가 기준에 못 미쳤다"는 판정입니다.

**왜 실패했는지 보려면?** 주의할 점이 있습니다. 게이트는 **후보의 작업 공간**에서 실행되고 그 공간은 실행 후
정리되며, 실패한 실행에는 적용할 패치도 남지 않습니다. 그래서 같은 검사 명령을 **원본에서** 다시 돌려도
후보의 실패가 아니라 원본의 원래 버그(`ZeroDivisionError`)가 보일 뿐입니다.

후보가 무엇을 만들었는지 보려면 **그 실행에만 게이트를 끄고** 같은 에이전트로 다시 실행해 `diff`를 읽습니다.
환경 변수 `SYNCMILL_GATE__COMMANDS='[]'`는 `syncmill.toml`을 고치지 않고 이번 실행에서만 게이트를 비웁니다.
(진짜 에이전트라면 다시 실행할 때 결과가 달라질 수 있고, 호출이 한 번 더 듭니다.)

In [27]:
print("실패한 실행의 diff 길이:", len(box.sm("diff", box.run_id(gated), show=False).stdout))

probe = box.sm("run", FIX_TASK, "--agent", "codex", extra_env={"SYNCMILL_GATE__COMMANDS": "[]"})
probe_id = box.run_id(probe)
probe_patch = box.sm("diff", probe_id).stdout
assert "+        return None" in probe_patch.splitlines()

실패한 실행의 diff 길이: 0


[syncmill] Executing task f77b8c95...
[syncmill] Agent: codex (forced, no fallback)

[codex] Completed in 0.02s
[syncmill] Files changed: calc.py

Return None for an empty list.
[syncmill] run: 46af4649-df29-44b3-a751-9d59fa75db96
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/46af4649-df29-44b3-a751-9d59fa75db96
[syncmill] review: syncmill diff 46af4649-df29-44b3-a751-9d59fa75db96
[syncmill] apply: syncmill apply 46af4649-df29-44b3-a751-9d59fa75db96


diff --git a/calc.py b/calc.py
index 3d50b59..e841a75 100644
--- a/calc.py
+++ b/calc.py
@@ -1,2 +1,4 @@
 def average(values):
+    if not values:
+        return None
     return sum(values) / len(values)


diff를 보면 codex가 빈 리스트에 `None`을 돌려주도록 고쳤다는 것이 보입니다. 검사는 `0`을 기대하므로 실패한
것이죠. 이렇게 원인을 확인했으면 이 결과는 쓰지 않으니 `discard`로 표시합니다.

In [28]:
box.sm("discard", probe_id)

Discarded run 46af4649-df29-44b3-a751-9d59fa75db96


<syncmill discard: exit 0>

이제 `--agent` 없이 실행해, codex가 게이트에 걸리면 kimi-code로 넘어가는지 봅시다.

In [29]:
prompts_before = len(box.prompts())
fixed = box.sm("run", FIX_TASK)
fixed_id = box.run_id(fixed)

called = [p.name.split("-")[0] for p in box.prompts()[prompts_before:]]
print("이번 실행에서 실제로 호출된 에이전트:", called)
assert called == ["codex", "kimi"]

[syncmill] Executing task 45f44973...
[syncmill] Agents: codex → kimi-code (priority fallback)

[kimi-code] Completed in 0.03s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: calc.py

Return 0 for an empty list.
[syncmill] run: ee75c4d5-3b46-4ac1-9c73-f4e906a543e0
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/ee75c4d5-3b46-4ac1-9c73-f4e906a543e0
[syncmill] review: syncmill diff ee75c4d5-3b46-4ac1-9c73-f4e906a543e0
[syncmill] apply: syncmill apply ee75c4d5-3b46-4ac1-9c73-f4e906a543e0
이번 실행에서 실제로 호출된 에이전트: ['codex', 'kimi']


**출력 읽는 법 — 대체 실행 + 게이트 통과**

- `Agents: codex → kimi-code (priority fallback)` — 이 순서로 시도한다는 뜻입니다.
- `[kimi-code] Completed ...` 다음의 `Quality gate: passed (1/1 checks passed)` — 최종 결과가 검사를 통과했습니다.
- `Files changed: calc.py` — `.gitignore` 덕분에 `__pycache__`가 섞이지 않았습니다.

**주의할 점**: 기본 출력에는 **최종 결과만** 나오고, 앞에서 게이트에 걸려 탈락한 codex의 시도는 보이지 않습니다.
위 셀에서 가짜 에이전트의 프롬프트 기록으로 codex도 실제로 호출됐다는 것을 확인했습니다. 진짜 에이전트라면 그만큼
사용량이 들었다는 뜻이니, "왜 이렇게 오래 걸렸지?" 싶을 때는 이 점을 떠올리세요.

결과를 검토합니다. 아직 적용하지 않고, 다음 장에서 이어서 다듬겠습니다.

In [30]:
patch = box.sm("diff", fixed_id).stdout
assert "+        return 0" in patch.splitlines()

diff --git a/calc.py b/calc.py
index 3d50b59..811045e 100644
--- a/calc.py
+++ b/calc.py
@@ -1,2 +1,4 @@
 def average(values):
+    if not values:
+        return 0
     return sum(values) / len(values)


## 11. 실무 응용 ②: 결과를 이어서 다듬기: `continue`

리뷰해 보니 고친 방향은 좋은데 **docstring이 없어** 아쉽습니다. 새로 `run`을 하면 에이전트가 처음부터 다시
상황을 파악해야 하지만, `continue`는 **같은 에이전트의 같은 대화 세션**을 이어서 추가 요청을 보냅니다.
"방금 한 작업에 이것만 더 해 줘"라고 말하는 것과 같습니다.

이번에도 가짜 kimi-code가 할 일을 미리 정해 둡니다(앞의 수정에 docstring을 더한 버전).

In [31]:
WITH_DOCSTRING = (
    "def average(values):\n"
    '    """Return the mean of values, or 0 for an empty list."""\n'
    "    if not values:\n"
    "        return 0\n"
    "    return sum(values) / len(values)\n"
)
box.set_scenario({"kimi-code": {"files": {"calc.py": WITH_DOCSTRING}, "message": "Added a docstring."}})

followup = box.sm("continue", fixed_id, "average에 docstring도 추가해 줘")
followup_id = box.run_id(followup)

sent = box.prompts()[-1].read_text(encoding="utf-8")
context, _, task_line = sent.rpartition("Task: ")
print("프롬프트 앞부분(공유 메모리 맥락):", "있음" if context.startswith("Context from previous work:") else "없음")
print("프롬프트 끝의 요청:", task_line)
assert task_line == "average에 docstring도 추가해 줘"


[kimi-code] Completed in 0.03s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: calc.py

Added a docstring.
[syncmill] run: c19c2ede-7a0d-4b03-8b33-458e9e1be731
[syncmill] artifact: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/c19c2ede-7a0d-4b03-8b33-458e9e1be731
[syncmill] review: syncmill diff c19c2ede-7a0d-4b03-8b33-458e9e1be731
[syncmill] apply: syncmill apply c19c2ede-7a0d-4b03-8b33-458e9e1be731
프롬프트 앞부분(공유 메모리 맥락): 있음
프롬프트 끝의 요청: average에 docstring도 추가해 줘


**출력 읽는 법 — `continue`**

- 새 `run:` id가 발급됩니다. 후속 결과도 독립된 실행 기록으로 남습니다.
- 게이트가 설정돼 있으므로 후속 결과도 `Quality gate: passed`를 거쳤습니다.
- 에이전트가 받은 프롬프트의 끝(`Task:` 뒤)은 **추가 요청 한 줄**이고, 앞부분에는 8장처럼 공유 메모리의
  관련 기록이 붙습니다. "calc.py를 이렇게 고쳤다"는 앞 작업의 맥락은 프롬프트에 다시 적지 않습니다. `continue`는
  앞 실행의 대화 세션(7장의 `provider_session_id`)을 이어서 열기 때문에, 에이전트가 그 대화를 이미 알고 있습니다.
  그래서 세션 id가 없는 실행에는 `continue`를 쓸 수 없습니다(`run has no resumable provider session`).

후속 결과의 diff와 실행 목록을 봅시다.

In [32]:
followup_patch = box.sm("diff", followup_id).stdout
box.sm("runs", "list")

status_of = {line.split()[0]: line.split()[1] for line in box.sm("runs", "list", show=False).stdout.splitlines()}
assert status_of[fixed_id] == "superseded"
assert status_of[followup_id] == "pending"

diff --git a/calc.py b/calc.py
index 3d50b59..656189e 100644
--- a/calc.py
+++ b/calc.py
@@ -1,2 +1,5 @@
 def average(values):
+    """Return the mean of values, or 0 for an empty list."""
+    if not values:
+        return 0
     return sum(values) / len(values)


c19c2ede-7a0d-4b03-8b33-458e9e1be731  pending      kimi-code  2026-10-14T11:55:38.660792+00:00
ee75c4d5-3b46-4ac1-9c73-f4e906a543e0  superseded   kimi-code  2026-10-14T11:55:37.149460+00:00
46af4649-df29-44b3-a751-9d59fa75db96  discarded    codex      2026-10-14T11:55:35.611661+00:00
59d84f72-32fd-4b21-8ae2-1b0286fa46b4  failed       none       2026-10-14T11:55:34.313378+00:00
b6b1fce4-2049-4fc6-a609-33c534721d80  discarded    kimi-code  2026-10-14T11:55:31.444950+00:00
df61caa9-5324-43b1-87f1-ae8b668d0137  failed       none       2026-10-14T11:55:30.165600+00:00
4357e49e-92c9-4c43-8187-4facfaa5efb8  pending      kimi-code  2026-10-14T11:55:28.742426+00:00
cf83dbf4-1767-4ec4-b991-02f2aa4b6688  applied      codex      2026-10-14T11:55:26.299571+00:00


- 후속 결과의 diff는 **앞의 수정 + docstring이 합쳐진 누적 변경**입니다. 원본 기준으로 보여 주므로, 이 diff
  하나만 검토하고 적용하면 됩니다.
- 앞의 결과는 `superseded`(대체됨)가 되어 더는 적용할 수 없습니다. 두 결과를 차례로 적용하다 꼬이는 일을 막아 줍니다.
- 이 두 가지는 **아직 적용하지 않은(`pending`) 결과를 이어 갈 때**의 동작입니다. 이미 `apply`한 결과를 이어 가면
  앞의 변경은 원본에 들어 있으므로, 후속 결과의 diff에는 새로 바뀐 부분만 나옵니다.

후속 결과를 적용하고, 원본에서 검사 스크립트를 직접 실행해 확인합니다.

In [33]:
box.sm("apply", followup_id)
after_fix = box.cmd("python3", "check_calc.py")
assert after_fix.returncode == 0
assert (box.repo / "calc.py").read_text(encoding="utf-8") == WITH_DOCSTRING
box.set_scenario(None)

Applied run c19c2ede-7a0d-4b03-8b33-458e9e1be731
Receipt: /private/tmp/syncmill-notebook.vkjgtt0g/state/runs/74833be1-ad47-40c9-ab98-55249486674f/runs/c19c2ede-7a0d-4b03-8b33-458e9e1be731/applications/20261014T115539904991Z.json
check_calc: all good


검사가 원본에서도 통과합니다. 여기까지가 실무에서 가장 자주 쓰는 흐름입니다.

1. 팀의 검사 명령을 `[gate]`에 등록해 둔다.
2. `run`으로 맡긴다 — 검사를 통과한 결과만 성공으로 받고, 실패하면 다음 에이전트가 시도한다.
3. `diff`로 검토하고, 아쉬운 점은 `continue`로 이어서 다듬는다.
4. `apply` 후 직접 확인하고 커밋한다.

**응용 아이디어**

| 하고 싶은 일 | 방법 |
|---|---|
| 린트·타입 검사도 통과해야 받기 | `commands = ["pytest -q", "ruff check ."]`처럼 명령을 여러 개 등록(모두 통과해야 성공). `&&`로 잇지 말고 항목을 나눕니다 |
| 여러 에이전트에게 동시에 시키고 하나 받기 | `-s compete` — 모두 동시에 실행(사용량은 에이전트 수만큼). 기본 선택 방식(`verified`)은 "가장 좋은 결과"가 아니라 **우선순위상 처음으로 게이트를 통과한 결과**를 고릅니다. 결과끼리 비교해 순위를 매기게 하려면 `--selection judge`(판정용 호출이 추가로 듦) |
| 큰 작업을 나눠 맡기기 | `-s decompose` — 한 에이전트가 작업을 조각으로 나누고, 조각들을 동시에 처리한 뒤 그 **요약**을 바탕으로 새 작업 공간에서 통합 구현을 다시 작성합니다(조각별 코드를 그대로 합치지는 않음) |
| 같은 작업을 단계별로 이어 맡기기 | `-s pipeline` — 앞 에이전트의 결과를 뒤 에이전트가 이어받음 |

전략별 자세한 설명은 [사용 안내서의 "전략 고르기"](../user-guide.ko.md#전략-고르기)에 있습니다.

## 12. (선택) 진짜 에이전트로 한 번 실행하기

여기까지는 전부 가짜 에이전트였습니다. 진짜 CLI로 같은 흐름을 확인하고 싶을 때만 아래 셀에서
`RUN_REAL = True`로 바꾸고 실행하세요. 기본값은 `False`라서 아무것도 호출하지 않습니다.

**실제 실행에서 달라지는 점**

- 몇 초가 아니라 **수십 초에서 몇 분**이 걸릴 수 있습니다.
- 같은 요청이라도 **결과가 매번 다를 수 있습니다.** 그래서 `diff` 검토가 더 중요합니다.
- 선택한 CLI(`codex`, `claude`, `kimi`)가 설치·로그인돼 있어야 하고, **실제 사용량/요금**이
  발생합니다. 호출은 정확히 한 번입니다(`--agent`로 하나만 지정).

**이 셀만 환경이 다릅니다.** 로그인 정보는 여러분의 홈 디렉터리에 있으므로, 이 셀은 **진짜
`HOME`과 환경**을 사용합니다. 그래서 memtomem은 여러분의 `~/.memtomem/config.json` 설정(예:
임베딩 제공자, 그에 따른 모델 다운로드)을 그대로 적용하고, 필요하다고 판단하면 그 설정 파일을
갱신할 수도 있습니다. 실행 결과·작업 공간·메모리 DB 경로는 여전히 샌드박스 안에 고정됩니다.

In [34]:
RUN_REAL = False      # 실제 CLI를 호출하려면 True로 바꾸세요
REAL_AGENT = "codex"  # "codex", "claude", "kimi-code" 중 하나

if RUN_REAL:
    real = box.sm(
        "run",
        "Create exactly one repository-root file named real-check.txt containing exactly ok "
        "followed by one newline. Do not modify, create, rename, delete, or commit any other file.",
        "--agent", REAL_AGENT, "--timeout", "120",
        real=True, check=False,  # 실패해도 출력을 보고 원인을 확인할 수 있게 멈추지 않습니다
    )
    if real.returncode == 0:
        box.sm("diff", box.run_id(real), real=True)
else:
    print("RUN_REAL = False 이므로 실제 호출을 건너뛰었습니다.")

RUN_REAL = False 이므로 실제 호출을 건너뛰었습니다.


## 13. 내 프로젝트에 적용하기

노트북에서 한 일을 실제 프로젝트에서 하면 이렇습니다. 처음 한 번은 이 순서대로 따라 하세요.

1. **에이전트 CLI 준비** — `codex`(또는 `claude`, `kimi`)를 설치하고, 그 CLI를 직접 한 번 열어
   로그인합니다.
2. **저장소로 이동** — syncmill은 커밋이 하나 이상 있는 Git 저장소에서만 동작합니다(`git init`만
   하고 커밋이 없으면 거부합니다). 병합 충돌이나 리베이스가 진행 중일 때도 실행을 거부합니다.
3. **`syncmill init`** → `syncmill.toml`과 `.syncmill/.gitignore`를 커밋합니다.
4. **`syncmill doctor`** → `Ready.`와 함께, 쓰려는 에이전트의 줄(설치 경로, 로그인 확인)이 정상인지
   봅니다. Kimi는 로그인을 확인하지 않으므로 `kimi`를 직접 한 번 실행해 로그인돼 있는지 확인하세요.
5. **`syncmill run --dry-run "<작업>" --agent codex`** → 호출 수를 확인합니다.
6. **`syncmill run "<작업>" --agent codex --timeout 600`** → 작고 위험이 낮은 작업으로 시작하세요.
7. **`syncmill diff <run-id>`** → 꼭 읽습니다.
8. **`syncmill apply <run-id>`** → 테스트를 돌려 보고 직접 커밋합니다.

익숙해지면 다음을 차례로 시도해 보세요.

- `--agent`를 빼고 실행 → 실패 시 대체 실행(9장)
- 결과를 자동으로 검사하는 **품질 게이트**(10장): `syncmill.toml`의 `[gate]`에 `commands = ["pytest -q"]`처럼
  팀의 검사 명령을 넣으면, 통과한 결과만 성공으로 칩니다. `.gitignore`에 검사 부산물(`__pycache__/` 등)이
  들어 있는지도 확인하세요.
- 여러 에이전트를 함께 쓰는 전략(`compete`, `pipeline`, `council`, `decompose`) —
  [사용 안내서의 "전략 고르기"](../user-guide.ko.md#전략-고르기)

**자주 막히는 곳**

| 증상 | 확인할 것 |
|---|---|
| `CLI tool 'codex' not found in PATH` | syncmill을 실행하는 같은 셸에서 `command -v codex`가 나오는지 |
| 에이전트가 인증 메시지와 함께 바로 실패 | 그 CLI를 직접 열어 로그인을 끝냈는지 |
| `Files changed`는 나오는데 내 폴더는 그대로 | 정상입니다. `diff`로 검토한 뒤 `apply`하세요 |
| 실행이 너무 비싸거나 시끄러움 | `--agent` 하나 → `route` → 동시 전략 순서로 넓히세요 |

더 많은 문제 해결 방법은 [사용 안내서](../user-guide.ko.md)의 "문제가 생겼을 때"에 있습니다.

## 14. 정리하기

샌드박스 폴더를 지웁니다. `cleanup()`은 **지금 이 노트북이 만든** 폴더임을 확인할 수 있을 때만
지웁니다(만들 때 `.syncmill-notebook` 표시 파일에 남긴 일회용 값과 비교합니다). 그 밖의 경로는
거부합니다.

> 중간에 커널을 재시작했다면 `box` 객체가 사라져 이 셀로는 지울 수 없습니다. 그때는
> `/tmp/syncmill-notebook.`으로 시작하는 폴더를 직접 지우면 됩니다.

In [35]:
root = box.root
box.cleanup()
print("삭제됨:", not root.exists())

삭제됨: True


## 정리: 이번에 쓴 명령

| 명령 | 언제 쓰나 |
|---|---|
| `syncmill init` | 저장소에서 처음 한 번 |
| `syncmill doctor` | 처음 설정할 때, 에이전트가 이상하게 실패할 때 |
| `syncmill run --dry-run "<작업>"` | 새 전략·설정을 쓰기 전에 호출 수 확인 |
| `syncmill run "<작업>" --agent <id> --timeout <초>` | 작업 맡기기 |
| `syncmill runs list` | 검토할 결과(`pending`)가 무엇인지 볼 때 |
| `syncmill diff <run-id>` | 적용하기 전에 항상 |
| `syncmill apply <run-id>` | 검토한 결과를 원본에 반영할 때 |
| `syncmill runs show <run-id>` | 누가·무엇을·언제 만들었는지 실행 기록을 볼 때 |
| `syncmill discard <run-id>` | 쓰지 않기로 한 결과를 표시할 때 |
| `syncmill continue <run-id> "<추가 요청>"` | 같은 에이전트 대화를 이어서 고치게 할 때 (미적용 결과를 이어 가면 diff는 누적) |
| `syncmill.toml`의 `[gate] commands` | 검사를 통과한 결과만 받고 싶을 때 |

실제 프로젝트에서 진짜 에이전트로 시험해 보기 전에는 [`/tmp`에서 안전하게 시험하기](../tmp-test-guide.ko.md)도
따라가 보세요.

다음 노트북: **[02. 전략 비교](02-strategies.ipynb)** — 여러 에이전트를 함께 쓰는 다섯 가지 방법을 같은 과제로 비교합니다.